# Session 10 · The same RAG with LangChain

**From Prompt to Production · EvisionC**

1. Set **ROUTE** in the first cell to your model route from Session 2 (`claude`, `gemini`, `openai` or `openrouter`).
2. Add your API key in Colab Secrets (key icon, left sidebar, Notebook access on) under the name for your route: `ANTHROPIC_API_KEY`, `GEMINI_API_KEY`, `OPENAI_API_KEY` or `OPENROUTER_API_KEY`.
3. Run the cells from top to bottom (Shift+Enter). **Lumo track:** everything runs as is. **Own case:** the comments say what to swap.

The guided player explains every cell.

In [ ]:
# Cell 1 · setup (the same in every lab)
!git clone -q --depth 1 https://github.com/lrebelfor/lumo-sandbox /content/lumo 2>/dev/null || git -C /content/lumo pull -q
!pip install -q anthropic openai chromadb langchain langchain-anthropic langchain-openai langchain-chroma
import sys; sys.path.insert(0, "/content/lumo")
import lumo
from course_llm import LLM

ROUTE = "claude"   # your model route from Session 2: "claude", "openai", "gemini", "openrouter" (or "ollama" outside Colab)
llm = LLM(ROUTE)
print("Connected:", llm)
from langchain_chroma import Chroma
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

# The same pipeline as Session 9, now with LangChain's standard building blocks.
class LocalEmbeddings:                          # Chroma's free local embedding model, in LangChain's shape
    def __init__(self): self.f = DefaultEmbeddingFunction()
    def embed_documents(self, texts): return [list(map(float, v)) for v in self.f(texts)]
    def embed_query(self, text): return self.embed_documents([text])[0]

texts = [p.strip() for _, text in lumo.handbook() for p in text.split("\n\n")[1:] if p.strip()]
store = Chroma.from_texts(texts, embedding=LocalEmbeddings())   # swap in Pinecone, pgvector... here
retriever = store.as_retriever(search_kwargs={"k": 3})
model = llm.langchain_model()                                   # your route, as a LangChain model

def answer(question):
    context = "\n".join(d.page_content for d in retriever.invoke(question))
    msg = model.invoke(f"Answer only from this context. If it isn't there, say you don't know.\n\n{context}\n\nQuestion: {question}")
    return msg.text

print(answer("Do I need to return a lamp that arrived broken?"))